In [1]:
import sys
import os
import importlib
sys.path.append(os.path.abspath(".."))

# Reload lại tools mới nhất
import app.ai.agent.procurement.tools.plan_detail_tool as pdt
import app.ai.agent.procurement.tools as pct_tools
importlib.reload(pdt)
importlib.reload(pct_tools)

from app.ai.agent.procurement.tools import (
    search_request_docs,
    get_request_doc_detail,
    check_plan_budget_detail,
    get_po_master_status,
)

# 1. Tra cứu Tờ trình
print("=== 1. TRA CỨU TỜ TRÌNH ===")
res1 = await search_request_docs.ainvoke({"so_to_trinh": "PUR/2025/000052"})
print(res1)

# 2. Chi tiết Tờ trình
print("\n=== 2. CHI TIẾT TỜ TRÌNH ===")
res2 = await get_request_doc_detail.ainvoke({"doc_identifier": "PUR/2025/000052"})
print(res2)

# 3. Kế hoạch ngân sách liên kết
print("\n=== 3. KẾ HOẠCH LIÊN KẾT ===")
res3 = await check_plan_budget_detail.ainvoke({"ma_ke_hoach": "0049/2025/TTr-0690905"})
print(res3)

# 4. Phiếu gọi hàng (PO)
print("\n=== 4. PHIẾU GỌI HÀNG (PO) ===")
res4 = await get_po_master_status.ainvoke({"ma_po": "PO069/26/0006"})
print(res4)

=== 1. TRA CỨU TỜ TRÌNH ===
Tìm thấy tổng cộng 1 tờ trình trên gAMSPro. Dưới đây là thông tin chi tiết:

1. Số Tờ trình: PUR/2025/000052
   - Mã hệ thống (REQ_ID): TRRD00000269630
   - Trạng thái duyệt: Đã duyệt (Chờ đầu mối mua sắm xử lý)
   - Người tạo: Trương Quang Bảo (Phòng: Phòng Hỗ trợ)
   - Đơn vị: Hội sở
   - Tổng tiền đề xuất: 50,000,000 VNĐ
   - Ngày lập: 2025-12-22T00:00:00
   - Trích yếu / Lý do: chu trương mua sắm
   - Mã Kế hoạch liên kết: 0049/2025/TTr-0690905

=== 2. CHI TIẾT TỜ TRÌNH ===
📋 CHI TIẾT TỜ TRÌNH: PUR/2025/000052
- Mã định danh hệ thống (REQ_ID): `TRRD00000269630`
- Người lập: Trương Quang Bảo (Phòng: Phòng Hỗ trợ — Hội sở)
- Đơn vị chịu chi phí: Hội sở — Phòng Hỗ trợ
- Tổng tiền đề xuất: 50,000,000 VNĐ
- Ngày tạo tờ trình: 2025-12-22T14:31:29
- Trạng thái: Đã duyệt (Chờ đầu mối mua sắm xử lý)
- Nội dung / Lý do: chu trương mua sắm
- Kế hoạch liên kết: 📌 `0049/2025/TTr-0690905`

=== 3. KẾ HOẠCH LIÊN KẾT ===
📊 THÔNG TIN KẾ HOẠCH NGÂN SÁCH LIÊN KẾT:
- Mã Kế h

In [ ]:
from app.ai.agent.procurement.state import ProcurementState
from langchain_core.messages import HumanMessage

# Khởi tạo state mẫu cho 1 phiên chat của Cán bộ Bảo
test_state: ProcurementState = {
    "messages": [HumanMessage(content="Cho tôi xem danh sách tờ trình mua sắm gần đây")],
    "user_name": "baotq",
    "session_id": "demo-session-01"
}

print("State khởi tạo thành công:", test_state)

In [ ]:
from app.ai.agent.procurement.prompts.registry import get_system_prompt, get_user_prompt

# 1. Nạp System Prompt
system_prompt = get_system_prompt()

# 2. Nạp User Prompt với lịch sử chat
user_prompt = get_user_prompt(
    query="Cho tôi xem chi tiết tờ trình PUR/2025/000052 đi",
    chat_history=[{"role": "user", "content": "Xem danh sách tờ trình"}]
)

print("=== USER PROMPT RENDERED ===")
print(user_prompt[:500])


In [ ]:
import sys
import os
import importlib
sys.path.append(os.path.abspath(".."))

from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from IPython.display import display, Markdown

# 1. Nạp Supervisor Graph & Procurement Graph
from app.ai.agent.supervisor.graph.graph import supervisor_graph
from app.ai.agent.procurement.graph.graph import procurement_graph

# ==============================================================================
# HÀM XỬ LÝ PIPELINE TOÀN DIỆN: SUPERVISOR ROUTER ➔ PROCUREMENT AGENT
# ==============================================================================
async def ask_enterprise_chatbot(user_query: str, chat_history: list[dict], user_name: str = "baotq"):
    print("=" * 80)
    print(f"👤 [CBNV BVBank ({user_name})]: {user_query}")
    print("=" * 80)
    
    # --------------------------------------------------------------------------
    # BƯỚC 1: SUPERVISOR AGENT PHÂN LOẠI INTENT
    # --------------------------------------------------------------------------
    sup_state = {
        "user_query": user_query,
        "chat_history": chat_history,
    }
    sup_res = await supervisor_graph.ainvoke(sup_state)
    route = sup_res.get("route")
    
    intent = getattr(route, "intent", "fallback")
    intent_val = intent.value if hasattr(intent, "value") else str(intent)
    confidence = getattr(route, "confidence", 1.0)
    reasoning = getattr(route, "reasoning", "")
    
    print(f"🤖 [SUPERVISOR ROUTER]")
    print(f"   ├─ Phân loại Intent : {intent_val.upper()}")
    print(f"   ├─ Độ tin cậy       : {confidence * 100:.1f}%")
    print(f"   └─ Lý do suy luận   : {reasoning}")
    
    # --------------------------------------------------------------------------
    # BƯỚC 2: ĐIỀU HƯỚNG SANG PHÂN HỆ PHÙ HỢP
    # --------------------------------------------------------------------------
    if intent_val in ("gamspro", "procurement"):
        print(f"\n🚀 [ROUTING] Chuyển tiếp request sang ➔ [PROCUREMENT AGENT (gAMSPro)]...")
        
        # Chuyển đổi chat_history sang danh sách BaseMessage cho ReAct Agent
        messages: list[BaseMessage] = []
        for msg in chat_history:
            if msg["role"] == "user":
                messages.append(HumanMessage(content=msg["content"]))
            elif msg["role"] == "assistant":
                messages.append(AIMessage(content=msg["content"]))
        messages.append(HumanMessage(content=user_query))
        
        # Thực thi Procurement LangGraph
        proc_state = {
            "messages": messages,
            "user_name": user_name,
            "session_id": "demo-bvbank-session",
        }
        proc_res = await procurement_graph.ainvoke(proc_state)
        
        # In các Tool Calls thực tế đã được AI kích hoạt
        for msg in proc_res.get("messages", []):
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"   🔧 [TOOL CALL ACTIVATED] ➔ {tc.get('name')}({tc.get('args')})")
        
        last_message = proc_res.get("messages", [])[-1]
        final_answer = last_message.content
        
        # Cập nhật lịch sử chat
        chat_history.append({"role": "user", "content": user_query})
        chat_history.append({"role": "assistant", "content": final_answer})
        
        print("\n📋 [PHẢN HỒI CỦA TRỢ LÝ MUA SẮM gAMSPro]:")
        display(Markdown(final_answer))
        return final_answer
    else:
        print(f"⚠️ Intent '{intent_val}' không thuộc phân hệ Mua sắm gAMSPro.")
        return ""

# ==============================================================================
# CHẠY THỬ NGHIỆM ĐA LƯỢT (5 LƯỢT HỘI THOẠI USER STORY DEMO)
# ==============================================================================
chat_session_history = []

# Lượt 1: Tra cứu danh sách tờ trình
await ask_enterprise_chatbot(
    user_query="Cho tôi xem danh sách các Tờ trình mua sắm tôi đã lập gần đây.",
    chat_history=chat_session_history,
    user_name="baotq"
)

# Lượt 2: Xem chi tiết 1 tờ trình cụ thể
await ask_enterprise_chatbot(
    user_query="Cho tôi xem chi tiết tờ trình PUR/2025/000052 đi.",
    chat_history=chat_session_history,
    user_name="baotq"
)

# Lượt 3: Đối soát Kế hoạch Ngân sách liên kết
await ask_enterprise_chatbot(
    user_query="Kiểm tra luôn Kế hoạch liên kết giúp tôi.",
    chat_history=chat_session_history,
    user_name="baotq"
)

# Lượt 4: Kiểm tra Đơn đặt hàng PO / Phiếu gọi hàng
await ask_enterprise_chatbot(
    user_query="Có PO nào được tạo từ tờ trình này chưa?",
    chat_history=chat_session_history,
    user_name="baotq"
)

# Lượt 5: Xin hướng dẫn 3 bước hành động tiếp theo (Next-Best-Action)
await ask_enterprise_chatbot(
    user_query="Bây giờ tôi cần làm gì để Tờ trình PUR/2025/000052 được duyệt?",
    chat_history=chat_session_history,
    user_name="baotq"
)


c:\Users\khanh\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


👤 [CBNV BVBank (baotq)]: Cho tôi xem danh sách các Tờ trình mua sắm tôi đã lập gần đây.
🤖 [SUPERVISOR ROUTER]
   ├─ Phân loại Intent : PROCUREMENT
   ├─ Độ tin cậy       : 98.0%
   └─ Lý do suy luận   : Câu hỏi yêu cầu xem danh sách các tờ trình mua sắm cá nhân, thuộc phạm vi phân hệ Kế hoạch mua sắm và quản lý tài sản cố định trong gAMSPro, đáp ứng tiêu chí phân loại 'procurement'.

🚀 [ROUTING] Chuyển tiếp request sang ➔ [PROCUREMENT AGENT (gAMSPro)]...
   🔧 [TOOL CALL ACTIVATED] ➔ search_request_docs({'so_to_trinh': None, 'type_job': 'DVKD', 'user_name': 'baotq'})

📋 [PHẢN HỒI CỦA TRỢ LÝ MUA SẮM gAMSPro]:


👤 [CBNV BVBank (baotq)]: Cho tôi xem chi tiết tờ trình PUR/2025/000052 đi.
🤖 [SUPERVISOR ROUTER]
   ├─ Phân loại Intent : PROCUREMENT
   ├─ Độ tin cậy       : 98.0%
   └─ Lý do suy luận   : Câu hỏi tra cứu chi tiết tờ trình mua sắm cụ thể qua mã PUR trên hệ thống gAMSPro, thuộc phạm vi procurement.

🚀 [ROUTING] Chuyển tiếp request sang ➔ [PROCUREMENT AGENT (gAMSPro)]...
   🔧 [TOOL CALL ACTIVATED] ➔ get_request_doc_detail({'user_name': 'baotq', 'doc_identifier': 'PUR/2025/000052'})

📋 [PHẢN HỒI CỦA TRỢ LÝ MUA SẮM gAMSPro]:


📊 **CHI TIẾT TỜ TRÌNH**  
- **Mã định danh hệ thống (REQ_ID):** `TRRD00000269630`  
- **Người lập:** Trương Quang Bảo (Phòng: Phòng Hỗ trợ — Hội sở)  
- **Đơn vị chịu chi phí:** Hội sở — Phòng Hỗ trợ  
- **Tổng tiền đề xuất:** 50,000,000 VNĐ  
- **Ngày tạo tờ trình:** 2025-12-22T14:31:29  
- **Trạng thái:** ✅ Đã duyệt (Chờ đầu mối mua sắm xử lý)  
- **Nội dung / Lý do:** Chu trương mua sắm  
- **Kế hoạch liên kết:** 📌 `0049/2025/TTr-0690905`  

💡 **Ghi chú:** Tờ trình đã được phê duyệt, cần tiếp tục xử lý bởi Đơn vị Chuyên môn (DVCM) để hoàn tất thủ tục.

👤 [CBNV BVBank (baotq)]: Kiểm tra luôn Kế hoạch liên kết giúp tôi.
🤖 [SUPERVISOR ROUTER]
   ├─ Phân loại Intent : PROCUREMENT
   ├─ Độ tin cậy       : 95.0%
   └─ Lý do suy luận   : Câu hỏi yêu cầu kiểm tra hạn mức kế hoạch liên kết trong hệ thống gAMSPro, thuộc phạm vi của phân hệ Kế hoạch mua sắm và kiểm soát ngân sách, nên được phân loại vào mục procurement.

🚀 [ROUTING] Chuyển tiếp request sang ➔ [PROCUREMENT AGENT (gAMSPro)]...

📋 [PHẢN HỒI CỦA TRỢ LÝ MUA SẮM gAMSPro]:


...

👤 [CBNV BVBank (baotq)]: Có PO nào được tạo từ tờ trình này chưa?
🤖 [SUPERVISOR ROUTER]
   ├─ Phân loại Intent : PROCUREMENT
   ├─ Độ tin cậy       : 95.0%
   └─ Lý do suy luận   : Câu hỏi tra cứu tồn tại của đơn đặt hàng PO liên kết với tờ trình mua sắm trên hệ thống gAMSPro.

🚀 [ROUTING] Chuyển tiếp request sang ➔ [PROCUREMENT AGENT (gAMSPro)]...
   🔧 [TOOL CALL ACTIVATED] ➔ get_po_master_status({'ma_po': 'TRRD00000269630'})

📋 [PHẢN HỒI CỦA TRỢ LÝ MUA SẮM gAMSPro]:


📊 **ĐƠN ĐẶT HÀNG PO (POs) ĐÃ TRẢ LỜI**  
- **Tờ trình gốc:** `TRRD00000269630`  
- **Đã tạo PO:** Không có  
- **Danh sách PO đang triển khai:**  
  1. `PO069/26/0006` - Giá 12,765 VNĐ (Đã duyệt)  
  2. `PO069/26/0004` - Giá 275,000 VNĐ (Đã duyệt)  
  3. `PO069/26/0005` - Giá 7,290,000 VNĐ (Đã duyệt)  
  4. `PO069/26/0003` - Giá 0 VNĐ (Đã duyệt)  
  5. `PO069/26/0002` - Giá 2,000,010 VNĐ (Đã duyệt)  

💡 **Ghi chú:** Tờ trình `TRRD00000269630` vẫn chưa được tạo thành Đơn đặt hàng PO. Cán bộ có thể kiểm tra tiến độ xử lý hồ sơ qua AI Chatbot để theo dõi.

👤 [CBNV BVBank (baotq)]: Bây giờ tôi cần làm gì để Tờ trình PUR/2025/000052 được duyệt?
🤖 [SUPERVISOR ROUTER]
   ├─ Phân loại Intent : PROCUREMENT
   ├─ Độ tin cậy       : 95.0%
   └─ Lý do suy luận   : Câu hỏi yêu cầu hướng dẫn các bước phê duyệt tờ trình mua sắm cụ thể, thuộc phạm vi quy trình nghiệp vụ procurement. Câu hỏi rõ ràng, không có yếu tố mơ hồ hoặc liên quan đến lĩnh vực khác.

🚀 [ROUTING] Chuyển tiếp request sang ➔ [PROCUREMENT AGENT (gAMSPro)]...

📋 [PHẢN HỒI CỦA TRỢ LÝ MUA SẮM gAMSPro]:


📊 **CÁC BƯỚC ĐỂ TỜ TRÌNH ĐƯỢC ĐÁNH GIÁ TUÂN THỦ**  
1. **Hoàn thiện Tờ trình mua sắm**  
   - Truy cập **"Manage Purchases"** → **"Add Request Details"**  
   - Nhập:  
     - **Mã Tờ trình:** `TRRD00000269630`  
     - **Loại hàng:** Chu trương mua sắm  
     - **Số lượng:** 1 đơn vị  
     - **Đơn giá:** 50,000,000 VNĐ  
     - **Đơn vị tính:** Cái  
     - **Mã Kế hoạch liên kết:** `0049/2025/TTr-0690905`  

2. **Xác nhận và gửi Tờ trình**  
   - Truy cập **"Request Approval"** → **"Submit for Approval"**  
   - Nhấn **"Submit"** để gửi Tờ trình đến **Đơn vị Chuyên môn (DVCM)**  

3. **Theo dõi tiến độ**  
   - Cán bộ có thể nhắn tin qua **AI Chatbot** để kiểm tra trạng thái xử lý hồ sơ.  

💡 **Ghi chú:** Tờ trình đã được phê duyệt, cần tiếp tục xử lý bởi DVCM để hoàn tất thủ tục.

'📊 **CÁC BƯỚC ĐỂ TỜ TRÌNH ĐƯỢC ĐÁNH GIÁ TUÂN THỦ**  \n1. **Hoàn thiện Tờ trình mua sắm**  \n   - Truy cập **"Manage Purchases"** → **"Add Request Details"**  \n   - Nhập:  \n     - **Mã Tờ trình:** `TRRD00000269630`  \n     - **Loại hàng:** Chu trương mua sắm  \n     - **Số lượng:** 1 đơn vị  \n     - **Đơn giá:** 50,000,000 VNĐ  \n     - **Đơn vị tính:** Cái  \n     - **Mã Kế hoạch liên kết:** `0049/2025/TTr-0690905`  \n\n2. **Xác nhận và gửi Tờ trình**  \n   - Truy cập **"Request Approval"** → **"Submit for Approval"**  \n   - Nhấn **"Submit"** để gửi Tờ trình đến **Đơn vị Chuyên môn (DVCM)**  \n\n3. **Theo dõi tiến độ**  \n   - Cán bộ có thể nhắn tin qua **AI Chatbot** để kiểm tra trạng thái xử lý hồ sơ.  \n\n💡 **Ghi chú:** Tờ trình đã được phê duyệt, cần tiếp tục xử lý bởi DVCM để hoàn tất thủ tục.'

In [3]:
await ask_enterprise_chatbot(
    user_query="Cho tôi xem chi tiết tờ trình PUR/2025/000052 đi.",
    chat_history=chat_session_history,
    user_name="baotq"
)

👤 [CBNV BVBank (baotq)]: Cho tôi xem chi tiết tờ trình PUR/2025/000052 đi.
🤖 [SUPERVISOR ROUTER]
   ├─ Phân loại Intent : PROCUREMENT
   ├─ Độ tin cậy       : 98.0%
   └─ Lý do suy luận   : Câu hỏi tra cứu chi tiết tờ trình mua sắm cụ thể qua mã PUR trên hệ thống gAMSPro, thuộc phạm vi quy trình xử lý tờ trình mua sắm trong phân hệ Procurement.

🚀 [ROUTING] Chuyển tiếp request sang ➔ [PROCUREMENT AGENT (gAMSPro)]...

📋 [PHẢN HỒI CỦA TRỢ LÝ MUA SẮM gAMSPro]:


📊 **CHI TIẾT TỜ TRÌNH**  
- **Mã định danh hệ thống (REQ_ID):** `TRRD00000269630`  
- **Người lập:** Trương Quang Bảo (Phòng: Phòng Hỗ trợ — Hội sở)  
- **Đơn vị chịu chi phí:** Hội sở — Phòng Hỗ trợ  
- **Tổng tiền đề xuất:** 50,000,000 VNĐ  
- **Ngày tạo tờ trình:** 2025-12-22T14:31:29  
- **Trạng thái:** ✅ Đã duyệt (Chờ đầu mối mua sắm xử lý)  
- **Nội dung / Lý do:** Chu trương mua sắm  
- **Kế hoạch liên kết:** 📌 `0049/2025/TTr-0690905`  

💡 **Ghi chú:** Tờ trình đã được phê duyệt, cần tiếp tục xử lý bởi Đơn vị Chuyên môn (DVCM) để hoàn tất thủ tục.  

📊 **ĐƠN ĐẶT HÀNG PO (POs) ĐÃ TRẢ LỜI**  
- **Tờ trình gốc:** `TRRD00000269630`  
- **Đã tạo PO:** Không có  
- **Danh sách PO đang triển khai:**  
  1. `PO069/26/0006` - Giá 12,765 VNĐ (Đã duyệt)  
  2. `PO069/26/0004` - Giá 275,000 VNĐ (Đã duyệt)  
  3. `PO069/26/0005` - Giá 7,290,000 VNĐ (Đã duyệt)  
  4. `PO069/26/0003` - Giá 0 VNĐ (Đã duyệt)  
  5. `PO069/26/0002` - Giá 2,000,010 VNĐ (Đã duyệt)  

💡 **Ghi chú:** Tờ trình `TRRD00000269630` vẫn chưa được tạo thành Đơn đặt hàng PO. Cán bộ có thể kiểm tra tiến độ xử lý hồ sơ qua AI Chatbot để theo dõi.

'📊 **CHI TIẾT TỜ TRÌNH**  \n- **Mã định danh hệ thống (REQ_ID):** `TRRD00000269630`  \n- **Người lập:** Trương Quang Bảo (Phòng: Phòng Hỗ trợ — Hội sở)  \n- **Đơn vị chịu chi phí:** Hội sở — Phòng Hỗ trợ  \n- **Tổng tiền đề xuất:** 50,000,000 VNĐ  \n- **Ngày tạo tờ trình:** 2025-12-22T14:31:29  \n- **Trạng thái:** ✅ Đã duyệt (Chờ đầu mối mua sắm xử lý)  \n- **Nội dung / Lý do:** Chu trương mua sắm  \n- **Kế hoạch liên kết:** 📌 `0049/2025/TTr-0690905`  \n\n💡 **Ghi chú:** Tờ trình đã được phê duyệt, cần tiếp tục xử lý bởi Đơn vị Chuyên môn (DVCM) để hoàn tất thủ tục.  \n\n📊 **ĐƠN ĐẶT HÀNG PO (POs) ĐÃ TRẢ LỜI**  \n- **Tờ trình gốc:** `TRRD00000269630`  \n- **Đã tạo PO:** Không có  \n- **Danh sách PO đang triển khai:**  \n  1. `PO069/26/0006` - Giá 12,765 VNĐ (Đã duyệt)  \n  2. `PO069/26/0004` - Giá 275,000 VNĐ (Đã duyệt)  \n  3. `PO069/26/0005` - Giá 7,290,000 VNĐ (Đã duyệt)  \n  4. `PO069/26/0003` - Giá 0 VNĐ (Đã duyệt)  \n  5. `PO069/26/0002` - Giá 2,000,010 VNĐ (Đã duyệt)  \n\n💡 **Ghi 